<a href="https://colab.research.google.com/github/joezein71/AIHC-5010-Winter-2026/blob/main/20260318_project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

Saving NLMCXR_reports.tgz to NLMCXR_reports.tgz
User uploaded file "NLMCXR_reports.tgz" with length 1112632 bytes


In [6]:
# List the contents of the current directory to see the extracted files
!ls -F

ecgen-radiology/  NLMCXR_reports.tgz  sample_data/


This command will show you all the files and directories that were extracted from `NLMCXR_reports.tgz`. If there's a specific folder or file you're interested in, let me know, and I can help you explore further.

Once you've uploaded the `NLMCXR_reports.tgz` file, run the following cell to extract its contents.

In [7]:
# Extract the contents of the .tgz file
!tar -xzf NLMCXR_reports.tgz

In [9]:
# List the first 5 contents of the 'ecgen-radiology' directory
!ls -F ecgen-radiology/ | head -n 5

1000.xml
1001.xml
1002.xml
1003.xml
1004.xml


In [13]:
import os
import glob
import pandas as pd
import xml.etree.ElementTree as ET

# List all XML files in the 'ecgen-radiology' directory
xml_files = glob.glob('ecgen-radiology/*.xml')

data_rows = []

for file_path in xml_files:
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()

        # Initialize extracted values with None or empty string
        # pmcid (for rowname/index) is ALWAYS the filename without .xml
        pmcid = os.path.basename(file_path).replace('.xml', '')
        uId = None
        docSource = None
        publisher = None
        title = None
        specialty = None
        subset = None
        comparison = ''
        indication = ''
        findings = ''
        impression = ''
        lastName = None
        foreName = None
        year = None
        month = None
        day = None
        parentImage_id = None
        caption = None
        panel_single_content = None

        # --- Extracting data based on XML structure assumptions ---
        # uId
        uId = root.findtext(".//MedlineCitation/PMID")

        # docSource
        docSource = root.findtext(".//MedlineCitation/Article/Journal/Title")
        if not docSource:
            docSource = root.findtext(".//MedlineCitation/Article/Journal/ISOAbbreviation")

        # publisher
        publisher = root.findtext(".//MedlineCitation/Article/Journal/Publisher/Name")

        # title
        title = root.findtext(".//MedlineCitation/Article/ArticleTitle")

        # specialty (e.g., from MeSH major topic or explicit tag)
        mesh_descriptor = root.find(".//MedlineCitation/MeshHeadingList/MeshHeading/DescriptorName[@MajorTopicYN='Y']")
        if mesh_descriptor is not None:
            specialty = mesh_descriptor.text
        else:
            specialty = root.findtext(".//MedlineCitation/Article/DataBankList/DataBank/AccessionNumberList/AccessionNumber") # A common place for specialty in some datasets

        # subset (e.g., from KeywordList or explicit tag)
        keyword_elem = root.find(".//MedlineCitation/KeywordList/Keyword")
        if keyword_elem is not None:
            subset = keyword_elem.text
        else:
            subset = root.findtext(".//MedlineCitation/Subset") # Check for direct <Subset> tag

        # COMPARISON, INDICATION, FINDINGS, IMPRESSION from AbstractText with Label attribute
        for abstract_text_elem in root.findall(".//MedlineCitation/Article/Abstract/AbstractText"):
            label = abstract_text_elem.get('Label')
            text_content = abstract_text_elem.text or ''
            if label == 'COMPARISON':
                comparison = text_content
            elif label == 'INDICATION':
                indication = text_content
            elif label == 'FINDINGS':
                findings = text_content
            elif label == 'IMPRESSION':
                impression = text_content

        # LastName, ForeName (first author)
        author_elem = root.find(".//MedlineCitation/Article/AuthorList/Author")
        if author_elem is not None:
            lastName = author_elem.findtext('LastName')
            foreName = author_elem.findtext('ForeName')

        # Year, Month, Day from PubDate
        pub_date_elem = root.find(".//MedlineCitation/Article/PubDate")
        if pub_date_elem is not None:
            year = pub_date_elem.findtext('Year')
            month = pub_date_elem.findtext('Month')
            day = pub_date_elem.findtext('Day')

        # parentImage id
        parent_image_elem = root.find(".//Image/ParentImage")
        if parent_image_elem is not None:
            parentImage_id = parent_image_elem.get('id')

        # caption
        caption = root.findtext(".//Image/Caption")

        # panel type="single" content
        panel_single_elem = root.find(".//Image/Panel[@type='single']")
        if panel_single_elem is not None:
            panel_single_content = panel_single_elem.text

        data_rows.append({
            'pmcId': pmcid,
            'uId': uId,
            'docSource': docSource,
            'publisher': publisher,
            'title': title,
            'specialty': specialty,
            'subset': subset,
            'COMPARISON': comparison,
            'INDICATION': indication,
            'FINDINGS': findings,
            'IMPRESSION': impression,
            'LastName': lastName,
            'ForeName': foreName,
            'Year': year,
            'Month': month,
            'Day': day,
            'parentImage_id': parentImage_id,
            'caption': caption,
            'panel_type_single_content': panel_single_content
        })

    except ET.ParseError as e:
        print(f"Error parsing XML file {file_path}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred while processing {file_path}: {e}")

# Create the DataFrame
df = pd.DataFrame(data_rows)

# Set pmcId as the index (rowname)
df = df.set_index('pmcId')

# Display the first 5 rows of the DataFrame
display(df.head())

,uId,docSource,publisher,title,specialty,subset,COMPARISON,INDICATION,FINDINGS,IMPRESSION,LastName,ForeName,Year,Month,Day,parentImage_id,caption,panel_type_single_content
pmcId,,,,,,,,,,,,,,,,,,
316,None,None,None,Indiana University Chest X-ray Collection,None,None,,preop inguinal hernia history of laryngeal cancer,,Heart size normal. Lungs are clear. 5 mm right...,Kohli,Marc,None,None,None,None,None,None
1281,None,None,None,Indiana University Chest X-ray Collection,None,None,"XXXX, XXXX","Headache, XXXX, and congestion.",Normal heart size and mediastinal contours. Lu...,No acute cardiopulmonary process. .,Kohli,Marc,None,None,None,None,None,None
3752,None,None,None,Indiana University Chest X-ray Collection,None,None,"XXXX, XXXX","XXXX, XXXX, EtOH",,1. Widened upper mediastinal silhouette. May r...,Kohli,Marc,None,None,None,None,None,None
1719,None,None,None,Indiana University Chest X-ray Collection,None,None,CT of the chest with high-resolution from XXXX.,"XXXX-year-old male, XXXX for mass/infiltrate",The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary abnormality.,Kohli,Marc,None,None,None,None,None,None
1386,None,None,None,Indiana University Chest X-ray Collection,None,None,,Dyspnea,The cardiac silhouette and upper mediastinum a...,1. No acute pulmonary infiltrate or effusion. ...,Kohli,Marc,None,None,None,None,None,None


In [18]:
# Print the dimensions (rows, columns) of the DataFrame
print(f"The DataFrame has {df.shape[0]} rows and {df.shape[1]} columns.")

The DataFrame has 3955 rows and 18 columns.


In [20]:
# Convert the pmcId index to numeric and then sort in ascending order
df_sorted = df.copy()
df_sorted.index = pd.to_numeric(df_sorted.index)
df_sorted = df_sorted.sort_index(ascending=True)

# Display the sorted DataFrame
display(df_sorted)

,uId,docSource,publisher,title,specialty,subset,COMPARISON,INDICATION,FINDINGS,IMPRESSION,LastName,ForeName,Year,Month,Day,parentImage_id,caption,panel_type_single_content
pmcId,,,,,,,,,,,,,,,,,,
1,None,None,None,Indiana University Chest X-ray Collection,None,None,None.,Positive TB test,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.,Kohli,Marc,None,None,None,None,None,None
2,None,None,None,Indiana University Chest X-ray Collection,None,None,None.,Preop bariatric surgery.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.,Kohli,Marc,None,None,None,None,None,None
3,None,None,None,Indiana University Chest X-ray Collection,None,None,,"rib pain after a XXXX, XXXX XXXX steps this XX...",,"No displaced rib fractures, pneumothorax, or p...",Kohli,Marc,None,None,None,None,None,None
4,None,None,None,Indiana University Chest X-ray Collection,None,None,None available,XXXX-year-old XXXX with XXXX.,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...,Kohli,Marc,None,None,None,None,None,None
5,None,None,None,Indiana University Chest X-ray Collection,None,None,,Chest and nasal congestion.,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.,Kohli,Marc,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,None,None,None,Indiana University Chest X-ray Collection,None,None,"XXXX, XXXX.","Nausea, vomiting x2 weeks. Dialysis patient.",The cardiomediastinal silhouette and pulmonary...,1. Interval resolution of bibasilar airspace d...,Kohli,Marc,None,None,None,None,None,None
3996,None,None,None,Indiana University Chest X-ray Collection,None,None,None.,,The lungs are clear. Heart size is normal. No ...,Clear lungs. No acute cardiopulmonary abnormal...,Kohli,Marc,None,None,None,None,None,None
3997,None,None,None,Indiana University Chest X-ray Collection,None,None,None available.,XXXX-year-old male with positive PPD.,"Heart size within normal limits. Small, nodula...","No acute findings, no evidence for active TB.",Kohli,Marc,None,None,None,None,None,None
